**OBJECTIVE**

The objective of this project is to build a news text classification
model using a Simple Recurrent Neural Network (RNN).
The model preprocesses and converts news articles into numerical sequences,
learns contextual information through word embeddings,
and classifies each article into one of four categories: World, Sports, Business, or Sci/Tech.
The project demonstrates the application of NLP and deep learning techniques for automated text
classification.

**Import Required Libraries**

Imports all necessary libraries for text preprocessing, dataset loading, tokenization, RNN model creation, and training.

In [1]:
import re
from datasets import load_dataset
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

**Load the AG News Dataset**


Downloads and loads the AG News dataset, then separates it into training and testing data.

In [3]:
print("Loading AG News Dataset...")

dataset = load_dataset("fancyzhx/ag_news")

train_texts = dataset["train"]["text"]
train_labels = dataset["train"]["label"]

test_texts = dataset["test"]["text"]
test_labels = dataset["test"]["label"]

print("Training Samples :", len(train_texts))
print("Testing Samples  :", len(test_texts))

Loading AG News Dataset...


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Training Samples : 120000
Testing Samples  : 7600


**Text Preprocessing (Cleaning)**


Converts text to lowercase and removes punctuation, numbers, and special characters to prepare clean input for the model.

In [4]:
def clean_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove punctuation, numbers, special characters
    text = re.sub(r'[^a-z\s]', '', text)

    return text

print("\nCleaning text...")

train_texts = [clean_text(text) for text in train_texts]
test_texts = [clean_text(text) for text in test_texts]

print("Sample Cleaned Text:")
print(train_texts[0])


Cleaning text...
Sample Cleaned Text:
wall st bears claw back into the black reuters reuters  shortsellers wall streets dwindlingband of ultracynics are seeing green again


**Tokenization**


Builds a vocabulary, converts words into integer tokens, and transforms text into numerical sequences.

In [5]:
print("\nTokenizing...")

VOCAB_SIZE = 10000

tokenizer = Tokenizer(num_words=VOCAB_SIZE)

# Learn vocabulary
tokenizer.fit_on_texts(train_texts)

# Convert text to integer sequences
X_train = tokenizer.texts_to_sequences(train_texts)
X_test = tokenizer.texts_to_sequences(test_texts)

print("Sample Sequence:")
print(X_train[0])


Tokenizing...
Sample Sequence:
[391, 324, 1525, 99, 54, 1, 812, 23, 23, 391, 1988, 4, 34, 3893, 737, 295]


**Sequence Padding**


Ensures every news article has the same length by padding shorter sequences and truncating longer ones.

In [6]:
print("\nPadding sequences...")

MAX_LENGTH = 50

X_train = pad_sequences(
    X_train,
    maxlen=MAX_LENGTH,
    padding='post',
    truncating='post'
)

X_test = pad_sequences(
    X_test,
    maxlen=MAX_LENGTH,
    padding='post',
    truncating='post'
)

print("Training Shape:", X_train.shape)
print("Testing Shape :", X_test.shape)


Padding sequences...
Training Shape: (120000, 50)
Testing Shape : (7600, 50)


**Label Encoding**


Converts categorical class labels into one-hot encoded vectors suitable for multi-class classification.

In [7]:
NUM_CLASSES = 4

y_train = to_categorical(train_labels, NUM_CLASSES)
y_test = to_categorical(test_labels, NUM_CLASSES)

print("\nSample Label:")
print(y_train[0])



Sample Label:
[0. 0. 1. 0.]


**Build the Simple RNN Model**

Builds a Sequential RNN model with an Embedding layer for word representations, a SimpleRNN layer to learn sequential patterns, and a Dense layer with Softmax activation for classifying news into four categories.

**Output**

Displays the model architecture. The model is shown as unbuilt with 0 parameters because the input shape is assigned only during training or when the model is explicitly built. The input_length warning is a deprecation warning and does not affect the model's execution.

In [8]:
print("\nBuilding Model...")

model = Sequential()

# Embedding Layer
model.add(
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=64,
        input_length=MAX_LENGTH
    )
)

# Simple RNN Layer
model.add(SimpleRNN(64))

# Output Layer
model.add(Dense(NUM_CLASSES, activation='softmax'))

model.summary()


Building Model...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

**Compile the Model**

Configures the model by specifying the optimizer (Adam), loss function (Categorical Crossentropy), and evaluation metric (Accuracy).

In [9]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

**Train the Model**

Trains the RNN on the training dataset for multiple epochs while monitoring validation performance.

In [10]:
print("\nTraining Started...\n")

history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2
)


Training Started...

Epoch 1/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 27s 32ms/step - accuracy: 0.8074 - loss: 0.5457 - val_accuracy: 0.8259 - val_loss: 0.5038
Epoch 2/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 41s 32ms/step - accuracy: 0.8921 - loss: 0.3432 - val_accuracy: 0.8606 - val_loss: 0.4176
Epoch 3/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 23s 30ms/step - accuracy: 0.9131 - loss: 0.2804 - val_accuracy: 0.8753 - val_loss: 0.3715
Epoch 4/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 25s 33ms/step - accuracy: 0.9262 - loss: 0.2353 - val_accuracy: 0.8757 - val_loss: 0.3788
Epoch 5/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 24s 32ms/step - accuracy: 0.9354 - loss: 0.2044 - val_accuracy: 0.8821 - val_loss: 0.3879


**Evaluate the Model**

Tests the trained model on unseen test data and reports the final test accuracy and loss.

In [11]:
print("\nEvaluating Model...")

loss, accuracy = model.evaluate(
    X_test,
    y_test
)

print("\nTest Accuracy: {:.2f}%".format(accuracy * 100))


Evaluating Model...
238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8893 - loss: 0.3592

Test Accuracy: 88.93%


**Predict News Category**

Takes a new news headline, preprocesses it, converts it into a sequence, performs prediction, and outputs the predicted news category (World, Sports, Business, or Sci/Tech).

In [12]:
label_names = [
    "World",
    "Sports",
    "Business",
    "Sci/Tech"
]

sample_news = [
    "Apple launches a new AI powered iPhone with advanced technology."
]

# Clean
sample_news = [clean_text(text) for text in sample_news]

# Convert to sequence
sample_sequence = tokenizer.texts_to_sequences(sample_news)

# Pad
sample_sequence = pad_sequences(
    sample_sequence,
    maxlen=MAX_LENGTH,
    padding='post'
)

# Predict
prediction = model.predict(sample_sequence)

predicted_class = prediction.argmax()

print("\nPrediction")
print("---------------------")
print("News :", sample_news[0])
print("Category :", label_names[predicted_class])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 213ms/step

Prediction
---------------------
News : apple launches a new ai powered iphone with advanced technology
Category : Sci/Tech
